# Credit Risk Temporal Drift Exploration

This notebook is the interactive frontend for the project code in `src/`.

It:
- loads and cleans the Lending Club dataset
- inspects the available columns
- builds a temporal train/test split
- trains the baseline logistic regression model
- evaluates temporal performance by year
- plots AUC and F1 over time

Use this notebook either:
1. locally with the dataset stored in `data/`, or
2. in Google Colab with the dataset stored in Google Drive.


## How this notebook fits into the repo

- `src/load_data.py` handles loading and cleaning
- `src/preprocess.py` selects features and builds preprocessing logic
- `src/temporal_split.py` creates temporal splits
- `src/models/logistic.py` trains the baseline logistic regression
- `src/evaluate.py` computes metrics and plots

This notebook imports those reusable modules instead of duplicating the project logic.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


## Configure repo path

If you are running this notebook from the repository's `notebooks/` folder, the cell below should work as-is.

If you are in Google Colab, you may need to edit `REPO_ROOT` manually after mounting Drive.


In [ ]:
cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

print("Current working directory:", cwd)
print("Repo root:", REPO_ROOT)


## Optional: mount Google Drive in Colab

Set `USE_GOOGLE_DRIVE = True` only if you are running in Colab and the repo or dataset is stored in Drive.


In [ ]:
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


## Optional: manually set repo root for Colab

Only use this if the automatic path detection above is wrong.

Example:
`REPO_ROOT = Path("/content/drive/MyDrive/CS-4365")`


In [ ]:
# Uncomment and edit if needed:
# REPO_ROOT = Path("/content/drive/MyDrive/CS-4365")
# if str(REPO_ROOT) not in sys.path:
#     sys.path.append(str(REPO_ROOT))

print("Using repo root:", REPO_ROOT)


## Import project modules


In [ ]:
from src.load_data import load_and_clean_dataset, summarize_dataset
from src.preprocess import (
    PreprocessConfig,
    get_model_feature_groups,
    summarize_feature_groups,
)
from src.temporal_split import (
    TemporalSplitConfig,
    make_temporal_split,
    describe_split,
    describe_test_years,
)
from src.models.logistic import fit_logistic_pipeline
from src.evaluate import (
    add_time_gap_column,
    evaluate_temporal_by_year,
    summarize_temporal_results,
    plot_temporal_metrics,
    save_temporal_metrics,
)


## Set dataset path

Pick one of the two patterns below.

### Local repo usage
Store the dataset in:
- `data/lending_club.parquet`, or
- `data/lending_club.csv`

### Google Drive / Colab usage
Use an absolute path like:
- `/content/drive/MyDrive/datasets/lending_club.parquet`


In [ ]:
# Example local path:
DATA_PATH = REPO_ROOT / "data" / "lending_club.parquet"

# Example Google Drive path:
# DATA_PATH = Path("/content/drive/MyDrive/datasets/lending_club.parquet")

print("DATA_PATH =", DATA_PATH)
print("Exists? ->", Path(DATA_PATH).exists())


## Load and clean dataset


In [ ]:
df = load_and_clean_dataset(DATA_PATH)
summarize_dataset(df)


## Inspect columns and a few rows

This is useful for confirming what is present in the curated dataset.


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nHead:")
display(df.head())


## Inspect target distribution and year distribution


In [ ]:
print("Target distribution:")
display(df["Default"].value_counts(dropna=False))

print("\nDefault rate:")
print(df["Default"].mean())

print("\nYear distribution:")
display(df["year"].value_counts().sort_index())


## Configure baseline preprocessing

For the Checkpoint 2 baseline:
- no text
- no ID
- drop zip code
- keep only categoricals with manageable cardinality


In [ ]:
preprocess_config = PreprocessConfig(
    include_text=False,
    include_id=False,
    drop_zip_code=True,
    max_categorical_cardinality=50,
)


## Build feature groups


In [ ]:
feature_groups = get_model_feature_groups(df, config=preprocess_config)
summarize_feature_groups(feature_groups)


## Inspect selected feature columns


In [ ]:
print("Feature columns:")
print(feature_groups["feature_cols"])

print("\nNumeric columns:")
print(feature_groups["numeric_cols"])

print("\nCategorical columns:")
print(feature_groups["categorical_cols"])

print("\nText columns:")
print(feature_groups["text_cols"])


## Configure temporal split

For the baseline experiment, train on loans up to 2014 and test on 2015-2018.


In [ ]:
split_config = TemporalSplitConfig(
    train_end_year=2014,
    test_years=(2015, 2016, 2017, 2018),
)

split_dict = make_temporal_split(df, config=split_config)


## Summarize the split


In [ ]:
split_summary_df = describe_split(split_dict)
split_summary_df


## Summarize the test years


In [ ]:
test_year_summary_df = describe_test_years(split_dict["test_df"])
test_year_summary_df


## Fit baseline logistic regression


In [ ]:
model = fit_logistic_pipeline(
    train_df=split_dict["train_df"],
    feature_groups=feature_groups,
)

print("Model fit complete.")


## Evaluate temporal performance by year


In [ ]:
results_df = evaluate_temporal_by_year(
    model=model,
    test_df=split_dict["test_df"],
    feature_groups=feature_groups,
)

results_df = add_time_gap_column(results_df, train_end_year=2014)
results_df


## Compact results table


In [ ]:
summarize_temporal_results(results_df)


## Plot performance vs test year


In [ ]:
plot_temporal_metrics(
    results_df=results_df,
    use_time_gap=False,
    title="Temporal Performance of Logistic Regression",
)


## Plot performance vs time gap


In [ ]:
plot_temporal_metrics(
    results_df=results_df,
    use_time_gap=True,
    title="Temporal Performance vs Time Gap",
)


## Save outputs into `results/`

This writes:
- `temporal_metrics.csv`
- `split_summary.csv`
- `test_year_summary.csv`
- `temporal_auc_f1.png`
- `temporal_auc_f1_time_gap.png`


In [ ]:
results_dir = REPO_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

save_temporal_metrics(results_df, results_dir / "temporal_metrics.csv")
split_summary_df.to_csv(results_dir / "split_summary.csv", index=False)
test_year_summary_df.to_csv(results_dir / "test_year_summary.csv", index=False)

plot_temporal_metrics(
    results_df=results_df,
    output_path=results_dir / "temporal_auc_f1.png",
    use_time_gap=False,
    title="Temporal Performance of Logistic Regression",
)

plot_temporal_metrics(
    results_df=results_df,
    output_path=results_dir / "temporal_auc_f1_time_gap.png",
    use_time_gap=True,
    title="Temporal Performance vs Time Gap",
)

print("Saved outputs to:", results_dir)
for p in sorted(results_dir.iterdir()):
    print("-", p.name)


## Optional next steps

After the baseline works, the same project structure can support:
- XGBoost
- MLP
- calibration metrics
- drift metrics such as PSI
- text embedding experiments using `title` and `desc`
